In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [ ]:
%pip install -q pinecone langchain-pinecone # --no-warn-script-location

# 1. knowledge base 구성을 위한 데이터 생성

In [5]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 4.05 s
Wall time: 4.09 s


180

In [6]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [ ]:
# # 임베딩 객체
# from dotenv import load_dotenv
# from langchain_upstage import UpstageEmbeddings
# load_dotenv()
# embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [3]:
len(embedding.embed_query("샘플"))

3072

In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [7]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# 데이터를 처음 저장할 때
index_name = 'tax-index-openai'
database = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)
# 업로드된 벡터 DB를 load할 때
# database = PineconeVectorStore(
#     embedding=embedding, # 질문을 임베딩하여 유사도 검색
#     index_name=index_name
# )

CPU times: total: 2.91 s
Wall time: 8.81 s


# 2. 답변 생성을 위한 Retrieval

In [8]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [14]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])
#print(retrieved_doc)

In [16]:
'55조' in retrieved_doc

False

# 3. 답변생성

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model='gpt-4.1-nano')

In [11]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [12]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [13]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉 5,000만원인 직장인의 소득세 계산은 다음과 같이 진행됩니다.  
(단, 기본공제 외 추가 공제 항목은 고려하지 않았으며, 근로소득공제 및 누진세율 적용 기준입니다.)

---

### **1. 근로소득 공제 적용**
- **근로소득 공제율** (2024년 기준):  
  - 5,000만원 이하: **70%** 공제  
  → 공제액 = 5,000만원 × 70% = **3,500만원**  
  → **공제 후 근로소득 금액** = 5,000만원 - 3,500만원 = **1,500만원**

---

### **2. 종합소득과세표준 계산**
- **인적공제(기본공제)** 적용 (본인 기준):  
  - 연간 기본공제액 = **150만원**  
  → **과세표준** = 1,500만원(근로소득) - 150만원(기본공제) = **1,350만원**  
  (*추가 공제(의료비, 보험료 등)가 있는 경우 과세표준이 더 감소*)

---

### **3. 산출세액 계산 (누진세율 적용)**
- 2024년 근로소득에 대한 **누진세율** (제55조):  
  | 과세표준 구간 | 세율 | 누진공제 |
  |----------------|------|----------|
  | 1,200만원 이하 | 6%   | 0원      |
  | 1,200만원 초과 ~ 4,600만원 이하 | 15% | 108만원 |

  - 1,350만원 중 1,200만원은 6% 적용, 나머지 150만원은 15% 적용:  
    - 1,200만원 × 6% = **72만원**  
    - 150만원 × 15% = **22.5만원**  
    → **산출세액** = 72만원 + 22.5만원 = **94.5만원**

---

### **4. 세액공제 적용**
- **근로소득 세액공제**: 산출세액의 **55%** 공제 (최대 74만원 한도)  
  → 공제액 = 94.5만원 × 55% ≈ **52만원**  
  → **납부세액** = 94.5만원 - 52만원 = **42.5만원**

---



# 4. Langchain

In [18]:
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import ChatUpstage
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
llm = ChatUpstage(model = 'solar-pro4')
# 2. vector DB load
pc = Pinecone()
index_name = 'tax-index-openai'
vectorStore = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)
# 3. Retriever 생성
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':3}) # 유사도가 높은 top-3추출
# # (2) search_type='mmr' : 후보 10개 중 서로 덜 겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr', 
#                                      search_kwargs={'k':3,
#                                                    'fetch_k':10,
#                                                    'lambda_mult':0.5}) # 1에 가까울수록 관련성 깊은 계산
# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold', 
#                                      search_kwargs={'k':3, 'score_threshold':0.4}) 
# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 문맥은 소득세법 조항만 포함하고 있어 연봉 5천만원에 대한 정확한 소득세 계산 금액(과세표준, 세율, 세액공제 등 반영 결과)을 도출할 수 없으므로 답변드릴 수 없습니다. 실제 소득세는 근로소득공제, 기본공제, 세액공제 등 여러 요소를 적용해야 하며, 정확한 금액은 근로소득 간이세액표 또는 종합소득세 계산을 통해 확정됩니다.
